# (T-)TEDOPA


## Efficient and accurate simulation. I get this

```bibtex
@article{10.1063/5.0223107,
    author = {Lacroix, Thibaut and Le Dé, Brieuc and Riva, Angela and Dunnett, Angus J. and Chin, Alex W.},
    title = {MPSDynamics.jl: Tensor network simulations for finite-temperature (non-Markovian) open quantum system dynamics},
    journal = {The Journal of Chemical Physics},
    volume = {161},
    number = {8},
    pages = {084116},
    year = {2024},
    month = {08},
    abstract = {The MPSDynamics.jl package provides an easy-to-use interface for performing open quantum systems simulations at zero and finite temperatures. The package has been developed with the aim of studying non-Markovian open system dynamics using the state-of-the-art numerically exact Thermalized-Time Evolving Density operator with Orthonormal Polynomials Algorithm based on environment chain mapping. The simulations rely on a tensor network representation of the quantum states as matrix product states (MPS) and tree tensor network states. Written in the Julia programming language, MPSDynamics.jl is a versatile open-source package providing a choice of several variants of the Time-Dependent Variational Principle method for time evolution (including novel bond-adaptive one-site algorithms). The package also provides strong support for the measurement of single and multi-site observables, as well as the storing and logging of data, which makes it a useful tool for the study of many-body physics. It currently handles long-range interactions, time-dependent Hamiltonians, multiple environments, bosonic and fermionic environments, and joint system–environment observables.},
    issn = {0021-9606},
    doi = {10.1063/5.0223107},
    url = {https://doi.org/10.1063/5.0223107},
    eprint = {https://pubs.aip.org/aip/jcp/article-pdf/doi/10.1063/5.0223107/20133487/084116\_1\_5.0223107.pdf},
}

@article{10.1063/5.0214051,
    author = {Le Dé, Brieuc and Jaouadi, Amine and Mangaud, Etienne and Chin, Alex W. and Desouter-Lecomte, Michèle},
    title = {Managing temperature in open quantum systems strongly coupled with structured environments},
    journal = {The Journal of Chemical Physics},
    volume = {160},
    number = {24},
    pages = {244102},
    year = {2024},
    month = {06},
    abstract = {In non-perturbative non-Markovian open quantum systems, reaching either low temperatures with the hierarchical equations of motion (HEOM) or high temperatures with the Thermalized Time Evolving Density Operator with Orthogonal Polynomials Algorithm (T-TEDOPA) formalism in Hilbert space remains challenging. We compare different ways of modeling the environment. Sampling the Fourier transform of the bath correlation function, also called temperature dependent spectral density, proves to be very effective. T-TEDOPA [Tamascelli et al., Phys. Rev. Lett. 123, 090402 (2019)] uses a linear chain of oscillators with positive and negative frequencies, while HEOM is based on the complex poles of an optimized rational decomposition of the temperature dependent spectral density [Xu et al., Phys. Rev. Lett. 129, 230601 (2022)]. Resorting to the poles of the temperature independent spectral density and of the Bose function separately is an alternative when the problem due to the huge number of Bose poles at low temperatures is circumvented. Two examples illustrate the effectiveness of the HEOM and T-TEDOPA approaches: a benchmark pure dephasing case and a two-bath model simulating the dynamics of excited electronic states coupled through a conical intersection. We show the efficiency of T-TEDOPA to simulate dynamics at a finite temperature by using either continuous spectral densities or only all the intramolecular oscillators of a linear vibronic model calibrated from ab initio data of a phenylene ethynylene dimer.},
    issn = {0021-9606},
    doi = {10.1063/5.0214051},
    url = {https://doi.org/10.1063/5.0214051},
    eprint = {https://pubs.aip.org/aip/jcp/article-pdf/doi/10.1063/5.0214051/20009074/244102\_1\_5.0214051.pdf},
}

@article{10.1063/1.3490188,
    author = {Chin, Alex W. and Rivas, Ángel and Huelga, Susana F. and Plenio, Martin B.},
    title = {Exact mapping between system-reservoir quantum models and semi-infinite discrete chains using orthogonal polynomials},
    journal = {Journal of Mathematical Physics},
    volume = {51},
    number = {9},
    pages = {092109},
    year = {2010},
    month = {09},
    abstract = {By using the properties of orthogonal polynomials, we present an exact unitary transformation that maps the Hamiltonian of a quantum system coupled linearly to a continuum of bosonic or fermionic modes to a Hamiltonian that describes a one-dimensional chain with only nearest-neighbor interactions. This analytical transformation predicts a simple set of relations between the parameters of the chain and the recurrence coefficients of the orthogonal polynomials used in the transformation and allows the chain parameters to be computed using numerically stable algorithms that have been developed to compute recurrence coefficients. We then prove some general properties of this chain system for a wide range of spectral functions and give examples drawn from physical systems where exact analytic expressions for the chain properties can be obtained. Crucially, the short-range interactions of the effective chain system permit these open-quantum systems to be efficiently simulated by the density matrix renormalization group methods.},
    issn = {0022-2488},
    doi = {10.1063/1.3490188},
    url = {https://doi.org/10.1063/1.3490188},
    eprint = {https://pubs.aip.org/aip/jmp/article-pdf/doi/10.1063/1.3490188/14758707/092109\_1\_online.pdf},
}


In [ ]:
"""Computing the chain map coefficients used in the (T-)TEDOPA algorithm.
"""


import numpy as np
from numpy.typing import NDArray
from typing import Optional, Callable
from tenso.libs.quantity import Quantity as __

from scipy.integrate import simpson, quad


class Tedopa:
    underflow = 1.0e-14

    @staticmethod
    def be_function(beta: float, w: NDArray) -> NDArray:
        return 0.5 + 0.5 / np.tanh(0.5 * beta * w)

    def __init__(self, w: NDArray, j: NDArray, beta: Optional[None], n_max: int):
        """
        The chain map coefficients used in the (T-)TEDOPA algorithm.

        Parameters
        ----------
        sd : Callable[[NDArray], NDArray]
            The spectral density function.
        beta : float
            The inverse temperature.
        n_max : int
            The maximum number of chain map coefficients to compute.
        ret
        """
        w = np.array(w)
        j = np.array(j)
        # Remove the zero-frequency component.
        if abs(w[0]) < self.underflow:
            w = w[1:]
            j = j[1:]

        if beta is None:
            self.frequency = w
            self.sd = j
            self.tsd = self.sd
        else:
            self.frequency = np.concatenate((-w[::-1], w))
            self.sd = np.concatenate((-j[::-1], j))
            self.tsd = self.be_function(beta, self.frequency) * self.sd
        self.beta = beta
        self.n_max = n_max

        # Coefficents for the orthogonal polynomials.
        # ip[n] = <p_n, p_n>
        self.ip = np.zeros(n_max)
        # ipx[n] = <x * p_n, p_n>
        self.ipx = np.zeros(n_max)
        # alpha[n] = <x * p_n, p_n> / <p_n, p_n>
        self.alpha = np.zeros(n_max)
        # beta[n] = <p_n, p_n> / <p_{n-1}, p_{n-1}>
        self.beta = np.zeros(n_max)
        self.generate_polynomials()
        return

    def c0(self) -> float:
        """
        Compute the zeroth-order coefficient.
        """
        return np.sqrt(self.ip[0])

    def chain_frequency(self) -> NDArray:
        """
        Compute the chain frequencies.
        """
        return self.alpha

    def chain_coupling(self) -> NDArray:
        """
        Compute the chain couplings.
        """
        return np.sqrt(self.beta)

    def chain_matrix(self) -> NDArray:
        """
        Compute the chain matrix.
        """
        return np.diag(self.chain_coupling()[1:], k=1) + \
            np.diag(self.chain_coupling()[1:], k=-1) + \
            np.diag(self.chain_frequency())

    def star_parameters(self) -> NDArray:
        e, v = np.linalg.eigh(self.chain_matrix())
        return e, self.c0() * v[:, 0]

    def int(self, f: NDArray) -> float:
        """
        Integrate a function with respect to the spectral density.
        """
        return simpson(f * self.tsd, self.frequency)

    def generate_polynomials(self) -> None:
        """
        Generate the orthogonal polynomials.
        """
        one = np.ones_like(self.frequency)
        # base case
        p_n2 = np.zeros_like(self.frequency)
        p_n1 = np.ones_like(self.frequency)
        self.ip[0] = self.int(p_n1 * p_n1)
        self.ipx[0] = self.int(self.frequency * p_n1 * p_n1)
        self.alpha[0] = self.ipx[0] / self.ip[0]
        self.beta[0] = 0.0
        for n in range(1, self.n_max):
            p_n = (self.frequency - self.alpha[n - 1] * one) * p_n1 - \
                self.beta[n - 1] * p_n2
            self.ip[n] = self.int(p_n * p_n)
            self.ipx[n] = self.int(self.frequency * p_n * p_n)
            self.alpha[n] = self.ipx[n] / self.ip[n]
            self.beta[n] = self.ip[n] / self.ip[n - 1]
            p_n2 = p_n1
            p_n1 = p_n
        return


if __name__ == "__main__":
    from matplotlib import pyplot as plt
    energy_unit = __(1000, '/cm').au
    freq_max = 45
    n_tedopa = 100
    beta = __(1 / 300.0, '/K').au * energy_unit
    print(beta)
    # beta = None
    re = 0.2
    gamma = 0.01
    print(gamma, re)

    print('Δ[J(w)/w]')
    for freq_max in [500]:
        n_space = int(freq_max * 10000)
        freq_space = (np.linspace(0, freq_max, n_space))[1:]
        # Drude-Lorentz spectral density
        # j = re / (2.0 * np.pi) * freq_space * \
        #     gamma / (freq_space**2 + gamma**2)
        # Ohmic spectral density
        j = re * 2 / np.sqrt(gamma * np.pi) * freq_space * \
            np.exp(-freq_space / gamma)
        tedopa_solver = Tedopa(freq_space, j, beta, n_tedopa)

        ww, gg = tedopa_solver.star_parameters()
        lc = simpson(tedopa_solver.tsd / tedopa_solver.frequency,
                     tedopa_solver.frequency)
        ld = sum(gg**2 / ww)
        d = lc - ld
        print(f'N={n_tedopa} | Fmax={freq_max} | {lc:.4} | {d:.4}')
        print(simpson(j/freq_space, freq_space))
        plt.plot(tedopa_solver.frequency,
                 tedopa_solver.sd, '-')
        plt.plot(ww, gg**2, 'x')
        for ni, (wi, gi) in enumerate(zip(ww, gg)):
            plt.text(wi, gi**2,
                     str(ni), color="black", fontsize=12)
        # plt.xlim(-3, 3)
        plt.show()